# 02 - Feature Correlation Analysis

This notebook analyzes correlations between features extracted from EncryptionGuard events.
We aggregate per-account features, compute correlation matrices, and visualize feature distributions.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure plotting
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

# Load events data
DATA_PATH = Path("../data/events.json")
with open(DATA_PATH) as f:
    events = json.load(f)

df = pd.DataFrame(events)

# Aggregate per-account features
account_col = None
for col in ["account_id", "merchant_id", "entity_id", "user_id"]:
    if col in df.columns:
        account_col = col
        break

if account_col:
    # Create per-account aggregations
    account_features = df.groupby(account_col).agg(
        event_count=("event_type", "count"),
        unique_event_types=("event_type", "nunique"),
    ).reset_index()

    # Add numeric column aggregations if available
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if col != account_col:
            account_features[f"{col}_mean"] = df.groupby(account_col)[col].mean().values
            account_features[f"{col}_sum"] = df.groupby(account_col)[col].sum().values

    print(f"Aggregated features for {len(account_features)} accounts")
    print(f"Feature columns: {list(account_features.columns)}")
    account_features.head()
else:
    print("No account column found. Using numeric columns directly.")
    account_features = df.select_dtypes(include=[np.number])
    account_features.head()

## Correlation Matrix

Computing pairwise Pearson correlations between all numeric features.

In [ ]:
# Select only numeric columns for correlation
numeric_features = account_features.select_dtypes(include=[np.number])

# Compute correlation matrix
corr_matrix = numeric_features.corr()

print(f"Correlation matrix shape: {corr_matrix.shape}")
print(f"\nTop correlated feature pairs:")

# Get top correlations (excluding self-correlations)
corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        corr_pairs.append({
            "feature_1": corr_matrix.columns[i],
            "feature_2": corr_matrix.columns[j],
            "correlation": corr_matrix.iloc[i, j]
        })

corr_df = pd.DataFrame(corr_pairs)
corr_df["abs_corr"] = corr_df["correlation"].abs()
print(corr_df.sort_values("abs_corr", ascending=False).head(10))

## Correlation Heatmap

Visual heatmap of the feature correlation matrix using seaborn.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

# Create heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
    vmin=-1,
    vmax=1
)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## Feature Distributions by Label

Comparing feature distributions across different labels to identify discriminative features.

In [ ]:
# Merge labels back if available
label_col = None
for col in ["label", "event_label", "is_suspicious", "fraud_flag"]:
    if col in df.columns:
        label_col = col
        break

if label_col and account_col:
    # Get label per account (most common label)
    account_labels = df.groupby(account_col)[label_col].agg(lambda x: x.mode()[0]).reset_index()
    merged = account_features.merge(account_labels, on=account_col, how="left")

    # Plot distributions for top numeric features
    plot_cols = [c for c in numeric_features.columns[:6]]  # Top 6 features

    if plot_cols:
        n_cols = 3
        n_rows = (len(plot_cols) + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
        axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 else axes

        for idx, col in enumerate(plot_cols):
            if idx < len(axes):
                for label in merged[label_col].unique():
                    subset = merged[merged[label_col] == label]
                    axes[idx].hist(subset[col].dropna(), alpha=0.5, label=str(label), bins=30)
                axes[idx].set_title(f"{col} by {label_col}")
                axes[idx].legend()

        # Hide unused axes
        for idx in range(len(plot_cols), len(axes)):
            axes[idx].set_visible(False)

        plt.tight_layout()
        plt.show()
    else:
        print("No numeric features to plot.")
else:
    print(f"Label column '{label_col}' or account column '{account_col}' not found.")
    print("Available columns:", list(df.columns))